# Weekly HMM Robustness Analysis

This notebook tests whether the weak incremental signal from the original weekly HMM is due to the univariate weekly-return specification. It keeps the main weekly HMM, baseline LSTM, and HMM-informed LSTM notebooks unchanged.

The robustness question is:

> Does a richer HMM, using volatility, momentum, and drawdown in addition to weekly returns, identify more informative market regimes than the simple univariate weekly-return HMM?


## Setup

The notebook uses the existing weekly modelling dataframe and the existing train/validation/test split. Scalers, HMMs, state labels, state-positive-return frequencies, and classification thresholds are all estimated without using test data.


In [ ]:
from __future__ import annotations

import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
    auc,
)
from sklearn.preprocessing import StandardScaler

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd()
INPUT_PATH = PROJECT_ROOT / "outputs" / "weekly" / "model_df_weekly.parquet"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "weekly_hmm_robustness"
PLOT_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_INPUT_COLS = [
    "Ticker",
    "Date",
    "Target_Date",
    "Split",
    "Weekly_Log_Return",
    "Rolling_Vol_12",
    "Momentum_12",
    "Drawdown_12",
    "Next_Week_Log_Return",
    "Target_State_Binary",
    "Target_State",
]

HMM_VARIANTS = {
    "univariate_2state": {
        "feature_cols": ["Weekly_Log_Return"],
        "n_components": 2,
    },
    "rich_2state": {
        "feature_cols": [
            "Weekly_Log_Return",
            "Rolling_Vol_12",
            "Momentum_12",
            "Drawdown_12",
        ],
        "n_components": 2,
    },
    "rich_3state": {
        "feature_cols": [
            "Weekly_Log_Return",
            "Rolling_Vol_12",
            "Momentum_12",
            "Drawdown_12",
        ],
        "n_components": 3,
    },
}

SEEDS = [42, 123, 2026]
THRESHOLDS = np.linspace(0.05, 0.95, 181)
LOW_PROBABILITY_STD_CUTOFF = 0.02

FEATURES_PATH = OUTPUT_DIR / "weekly_hmm_robustness_features.parquet"
STATE_SUMMARY_PATH = OUTPUT_DIR / "weekly_hmm_robustness_state_summary.csv"
TRANSITION_PATH = OUTPUT_DIR / "weekly_hmm_robustness_transition_matrices.csv"
EXPECTED_DURATION_PATH = OUTPUT_DIR / "weekly_hmm_robustness_expected_durations.csv"
MODEL_DIAGNOSTICS_PATH = OUTPUT_DIR / "weekly_hmm_robustness_model_diagnostics.csv"
HMM_ONLY_RESULTS_PATH = OUTPUT_DIR / "weekly_hmm_robustness_hmm_only_results.csv"
HMM_ONLY_PREDICTIONS_PATH = OUTPUT_DIR / "weekly_hmm_robustness_hmm_only_predictions.csv"
THRESHOLD_SEARCH_PATH = OUTPUT_DIR / "weekly_hmm_robustness_threshold_search.csv"
VARIANT_COMPARISON_PATH = OUTPUT_DIR / "weekly_hmm_robustness_variant_comparison.csv"
ROLLING_ROC_PATH = OUTPUT_DIR / "weekly_hmm_robustness_rolling_roc.csv"
SUBPERIOD_METRICS_PATH = OUTPUT_DIR / "weekly_hmm_robustness_subperiod_metrics.csv"

EXPECTED_OUTPUT_FILES = [
    FEATURES_PATH,
    STATE_SUMMARY_PATH,
    TRANSITION_PATH,
    EXPECTED_DURATION_PATH,
    MODEL_DIAGNOSTICS_PATH,
    HMM_ONLY_RESULTS_PATH,
    HMM_ONLY_PREDICTIONS_PATH,
    THRESHOLD_SEARCH_PATH,
    VARIANT_COMPARISON_PATH,
    ROLLING_ROC_PATH,
    SUBPERIOD_METRICS_PATH,
]

EXPECTED_PLOT_FILES = [
    PLOT_DIR / "weekly_hmm_robustness_variant_comparison.png",
    PLOT_DIR / "weekly_hmm_robustness_delta_vs_univariate.png",
    PLOT_DIR / "weekly_hmm_robustness_state_mean_vol_map.png",
    PLOT_DIR / "weekly_hmm_robustness_expected_durations.png",
    PLOT_DIR / "weekly_hmm_robustness_positive_return_prob_over_time.png",
    PLOT_DIR / "weekly_hmm_robustness_roc_curves_by_variant.png",
    PLOT_DIR / "weekly_hmm_robustness_rolling_roc_52.png",
    PLOT_DIR / "weekly_hmm_robustness_rolling_roc_104.png",
    PLOT_DIR / "weekly_hmm_robustness_subperiod_roc.png",
]

saved_plot_paths = []


def save_plot(filename: str) -> Path:
    path = PLOT_DIR / filename
    plt.savefig(path, dpi=300, bbox_inches="tight")
    saved_plot_paths.append(path)
    print(f"Saved plot: {path}")
    return path


print("Input path:", INPUT_PATH)
print("Output directory:", OUTPUT_DIR)
print("HMM variants:", list(HMM_VARIANTS))


## Load Weekly Data

In [ ]:
weekly_df = pd.read_parquet(INPUT_PATH)
missing_cols = [col for col in REQUIRED_INPUT_COLS if col not in weekly_df.columns]
if missing_cols:
    raise ValueError(f"Missing required input columns: {missing_cols}")

weekly_df = weekly_df[REQUIRED_INPUT_COLS].copy()
weekly_df["Date"] = pd.to_datetime(weekly_df["Date"])
weekly_df["Target_Date"] = pd.to_datetime(weekly_df["Target_Date"])
weekly_df = weekly_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

tickers = sorted(weekly_df["Ticker"].unique().tolist())
split_counts = weekly_df.groupby(["Ticker", "Split"]).size().unstack(fill_value=0)

display(split_counts)
display(weekly_df.head())


## HMM Helpers

In [ ]:
def logsumexp_np(values: np.ndarray, axis: int | None = None) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    max_values = np.max(values, axis=axis, keepdims=True)
    result = max_values + np.log(np.sum(np.exp(values - max_values), axis=axis, keepdims=True))
    if axis is not None:
        result = np.squeeze(result, axis=axis)
    return result


def filtered_state_probabilities(model: GaussianHMM, X: np.ndarray) -> np.ndarray:
    """Return causal filtered probabilities P(S_t | Y_1:t)."""
    X = np.asarray(X, dtype=float)
    log_likelihood = model._compute_log_likelihood(X)
    eps = np.finfo(float).tiny
    log_startprob = np.log(np.clip(model.startprob_, eps, 1.0))
    log_transmat = np.log(np.clip(model.transmat_, eps, 1.0))

    log_alpha = np.empty_like(log_likelihood)
    log_alpha[0] = log_startprob + log_likelihood[0]
    log_alpha[0] -= logsumexp_np(log_alpha[0])

    for t in range(1, len(X)):
        log_pred = logsumexp_np(log_alpha[t - 1][:, None] + log_transmat, axis=0)
        log_alpha[t] = log_pred + log_likelihood[t]
        log_alpha[t] -= logsumexp_np(log_alpha[t])

    filtered_probs = np.exp(log_alpha)
    filtered_probs /= filtered_probs.sum(axis=1, keepdims=True)
    return filtered_probs


def expected_duration(stay_probability: float) -> float:
    if np.isclose(1 - stay_probability, 0):
        return np.inf
    return float(1 / (1 - stay_probability))


def n_hmm_parameters(n_components: int, n_features: int) -> int:
    return int(
        (n_components - 1)
        + n_components * (n_components - 1)
        + n_components * n_features
        + n_components * n_features * (n_features + 1) / 2
    )


def fit_best_hmm(X_train_scaled: np.ndarray, n_components: int, seeds: list[int]) -> tuple[GaussianHMM, int, float]:
    fitted_candidates = []
    failures = []
    for seed in seeds:
        model = GaussianHMM(
            n_components=n_components,
            covariance_type="full",
            n_iter=500,
            tol=1e-4,
            random_state=seed,
            min_covar=1e-4,
        )
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model.fit(X_train_scaled)
            train_log_likelihood = float(model.score(X_train_scaled))
            if not bool(model.monitor_.converged):
                failures.append((seed, "did not converge"))
                continue
            fitted_candidates.append((train_log_likelihood, seed, model))
        except Exception as exc:
            failures.append((seed, repr(exc)))

    if not fitted_candidates:
        raise RuntimeError(f"All HMM seeds failed or did not converge. Failures: {failures}")

    fitted_candidates.sort(key=lambda item: item[0], reverse=True)
    best_log_likelihood, best_seed, best_model = fitted_candidates[0]
    return best_model, best_seed, best_log_likelihood


def state_statistics(
    train_df: pd.DataFrame,
    train_states: np.ndarray,
    n_components: int,
) -> pd.DataFrame:
    state_df = train_df.copy()
    state_df["HMM_State"] = train_states.astype(int)
    state_df["Current_Return_Positive"] = (state_df["Weekly_Log_Return"] > 0).astype(int)
    rows = []
    for state in range(n_components):
        group = state_df.loc[state_df["HMM_State"] == state]
        rows.append(
            {
                "HMM_State": state,
                "Train_State_Count": int(len(group)),
                "Train_Mean_Weekly_Log_Return": float(group["Weekly_Log_Return"].mean()) if len(group) else np.nan,
                "Train_Std_Weekly_Log_Return": float(group["Weekly_Log_Return"].std()) if len(group) > 1 else np.nan,
                "Train_Positive_Return_Probability": float(group["Current_Return_Positive"].mean()) if len(group) else 0.5,
            }
        )
    return pd.DataFrame(rows)


def label_two_state_hmm(stats_df: pd.DataFrame) -> tuple[dict[int, str], str]:
    sorted_states = stats_df.sort_values(
        ["Train_Mean_Weekly_Log_Return", "HMM_State"],
        ascending=[False, True],
    )["HMM_State"].tolist()
    state_to_label = {
        int(sorted_states[0]): "Bullish",
        int(sorted_states[-1]): "Bearish",
    }
    mean_gap = float(stats_df["Train_Mean_Weekly_Log_Return"].max() - stats_df["Train_Mean_Weekly_Log_Return"].min())
    vol_gap = float(stats_df["Train_Std_Weekly_Log_Return"].max() - stats_df["Train_Std_Weekly_Log_Return"].min())
    if vol_gap > abs(mean_gap):
        diagnostic = "Volatility difference is larger than mean-return difference; regimes may be calm/stress rather than strictly bullish/bearish."
    else:
        diagnostic = "Mean-return difference is larger than volatility difference; bullish/bearish labels are comparatively interpretable."
    return state_to_label, diagnostic


def label_three_state_hmm(stats_df: pd.DataFrame) -> tuple[dict[int, str], str]:
    working = stats_df.copy()
    working["Train_Std_Weekly_Log_Return"] = working["Train_Std_Weekly_Log_Return"].fillna(working["Train_Std_Weekly_Log_Return"].median())
    lowest_mean_state = int(working.sort_values(["Train_Mean_Weekly_Log_Return", "HMM_State"]).iloc[0]["HMM_State"])
    highest_vol_state = int(working.sort_values(["Train_Std_Weekly_Log_Return", "HMM_State"], ascending=[False, True]).iloc[0]["HMM_State"])

    candidate_states = sorted({lowest_mean_state, highest_vol_state})
    stress_state = int(
        working.loc[working["HMM_State"].isin(candidate_states)]
        .sort_values(["Train_Std_Weekly_Log_Return", "Train_Mean_Weekly_Log_Return"], ascending=[False, True])
        .iloc[0]["HMM_State"]
    )
    calm_state = int(
        working.loc[working["HMM_State"] != stress_state]
        .sort_values(["Train_Mean_Weekly_Log_Return", "Train_Std_Weekly_Log_Return"], ascending=[False, True])
        .iloc[0]["HMM_State"]
    )
    neutral_state = int([state for state in working["HMM_State"].astype(int).tolist() if state not in {stress_state, calm_state}][0])

    state_to_label = {
        calm_state: "High_Return_Calm",
        neutral_state: "Neutral_Transition",
        stress_state: "Low_Return_Stress",
    }
    note = (
        "Three-state labels are descriptive: stress is assigned to the lowest-mean or highest-volatility state, "
        "calm is the highest-mean remaining state, and the remaining state is neutral/transition."
    )
    return state_to_label, note


def binary_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> dict[str, float]:
    y_true = np.asarray(y_true).reshape(-1).astype(int)
    y_prob = np.asarray(y_prob).reshape(-1).astype(float)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "share_predicted_bullish": float(np.mean(y_pred)),
        "share_actual_bullish": float(np.mean(y_true)),
        "probability_mean": float(np.mean(y_prob)),
        "probability_std": float(np.std(y_prob)),
        "probability_min": float(np.min(y_prob)),
        "probability_max": float(np.max(y_prob)),
    }


def select_validation_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> tuple[float, pd.DataFrame]:
    rows = []
    for threshold in THRESHOLDS:
        metrics = binary_metrics(y_true, y_prob, threshold=float(threshold))
        rows.append(
            {
                "threshold": float(threshold),
                "balanced_accuracy": metrics["balanced_accuracy"],
                "accuracy": metrics["accuracy"],
                "f1": metrics["f1"],
                "cohen_kappa": metrics["cohen_kappa"],
                "roc_auc": metrics["roc_auc"],
                "predicted_bullish_share": metrics["share_predicted_bullish"],
                "distance_from_0_5": abs(float(threshold) - 0.5),
            }
        )
    threshold_df = pd.DataFrame(rows)
    best_row = threshold_df.sort_values(
        ["balanced_accuracy", "distance_from_0_5", "threshold"],
        ascending=[False, True, True],
    ).iloc[0]
    return float(best_row["threshold"]), threshold_df.drop(columns="distance_from_0_5")


## Fit Robustness HMM Variants

In [ ]:
feature_frames = []
state_summary_rows = []
transition_rows = []
duration_rows = []
diagnostic_rows = []
scaler_rows = []

for ticker in tickers:
    ticker_df = weekly_df.loc[weekly_df["Ticker"] == ticker].sort_values("Date").reset_index(drop=True).copy()
    train_mask = ticker_df["Split"] == "train"
    train_df = ticker_df.loc[train_mask].copy()
    if len(train_df) < 50:
        raise ValueError(f"{ticker} has too few training rows for HMM fitting: {len(train_df)}")

    for variant, spec in HMM_VARIANTS.items():
        feature_cols = spec["feature_cols"]
        n_components = int(spec["n_components"])
        feature_set = ", ".join(feature_cols)

        scaler = StandardScaler()
        X_train = train_df[feature_cols].to_numpy(dtype=float)
        X_all = ticker_df[feature_cols].to_numpy(dtype=float)
        scaler.fit(X_train)
        X_train_scaled = scaler.transform(X_train)
        X_all_scaled = scaler.transform(X_all)

        for feature, mean, scale in zip(feature_cols, scaler.mean_, scaler.scale_):
            scaler_rows.append(
                {
                    "Ticker": ticker,
                    "Variant": variant,
                    "Feature": feature,
                    "Train_Mean": float(mean),
                    "Train_Scale": float(scale),
                    "Fitted_On": "train",
                }
            )

        hmm, selected_seed, train_log_likelihood = fit_best_hmm(X_train_scaled, n_components, SEEDS)
        filtered_probs = filtered_state_probabilities(hmm, X_all_scaled)
        next_probs = filtered_probs @ hmm.transmat_
        hard_states = filtered_probs.argmax(axis=1).astype(int)
        train_states = hard_states[np.flatnonzero(train_mask.to_numpy())]

        stats_df = state_statistics(train_df, train_states, n_components)
        if n_components == 2:
            state_to_label, label_note = label_two_state_hmm(stats_df)
        elif n_components == 3:
            state_to_label, label_note = label_three_state_hmm(stats_df)
        else:
            raise ValueError(f"Unexpected n_components: {n_components}")

        state_positive_probs = dict(
            zip(stats_df["HMM_State"].astype(int), stats_df["Train_Positive_Return_Probability"].astype(float))
        )
        positive_prob_next = np.zeros(len(ticker_df), dtype=float)
        for state in range(n_components):
            positive_prob_next += next_probs[:, state] * state_positive_probs.get(state, 0.5)

        n_features = len(feature_cols)
        n_params = n_hmm_parameters(n_components, n_features)
        n_train = len(train_df)
        aic = 2 * n_params - 2 * train_log_likelihood
        bic = math.log(n_train) * n_params - 2 * train_log_likelihood

        variant_features = ticker_df[
            [
                "Ticker",
                "Date",
                "Target_Date",
                "Split",
                "Weekly_Log_Return",
                "Next_Week_Log_Return",
                "Target_State_Binary",
                "Target_State",
            ]
        ].copy()
        variant_features.insert(3, "Variant", variant)
        variant_features.insert(4, "n_components", n_components)
        variant_features["HMM_State"] = hard_states
        variant_features["HMM_Regime_Label"] = variant_features["HMM_State"].map(state_to_label)
        variant_features["HMM_Positive_Return_Prob_Next"] = positive_prob_next

        if n_components == 2:
            label_to_state = {label: state for state, label in state_to_label.items()}
            variant_features["HMM_Bullish_Prob_Next"] = next_probs[:, label_to_state["Bullish"]]
            variant_features["HMM_Bearish_Prob_Next"] = next_probs[:, label_to_state["Bearish"]]
            for state in range(3):
                variant_features[f"HMM_State_{state}_Prob_Next"] = np.nan
        else:
            variant_features["HMM_Bullish_Prob_Next"] = np.nan
            variant_features["HMM_Bearish_Prob_Next"] = np.nan
            for state in range(3):
                variant_features[f"HMM_State_{state}_Prob_Next"] = next_probs[:, state]

        feature_frames.append(variant_features)

        for state in range(n_components):
            stay_probability = float(hmm.transmat_[state, state])
            state_stats = stats_df.loc[stats_df["HMM_State"] == state].iloc[0]
            state_summary_rows.append(
                {
                    "Ticker": ticker,
                    "Variant": variant,
                    "n_components": n_components,
                    "feature_set": feature_set,
                    "selected_seed": selected_seed,
                    "HMM_State": state,
                    "HMM_Regime_Label": state_to_label[state],
                    "Train_State_Count": int(state_stats["Train_State_Count"]),
                    "Train_Mean_Weekly_Log_Return": state_stats["Train_Mean_Weekly_Log_Return"],
                    "Train_Std_Weekly_Log_Return": state_stats["Train_Std_Weekly_Log_Return"],
                    "Train_Positive_Return_Probability": state_stats["Train_Positive_Return_Probability"],
                    "stay_probability": stay_probability,
                    "expected_duration": expected_duration(stay_probability),
                    "Label_Rule_Note": label_note,
                }
            )
            duration_rows.append(
                {
                    "Ticker": ticker,
                    "Variant": variant,
                    "HMM_State": state,
                    "HMM_Regime_Label": state_to_label[state],
                    "stay_probability": stay_probability,
                    "expected_duration": expected_duration(stay_probability),
                }
            )
            for to_state in range(n_components):
                transition_rows.append(
                    {
                        "Ticker": ticker,
                        "Variant": variant,
                        "From_State": state,
                        "To_State": to_state,
                        "From_Label": state_to_label[state],
                        "To_Label": state_to_label[to_state],
                        "Transition_Probability": float(hmm.transmat_[state, to_state]),
                    }
                )

        diagnostic_rows.append(
            {
                "Ticker": ticker,
                "Variant": variant,
                "n_components": n_components,
                "n_features": n_features,
                "feature_set": feature_set,
                "selected_seed": selected_seed,
                "train_log_likelihood": train_log_likelihood,
                "AIC": aic,
                "BIC": bic,
                "converged": bool(hmm.monitor_.converged),
                "iterations": int(hmm.monitor_.iter),
                "n_parameters": n_params,
                "n_observations_train": n_train,
                "Scaler_Fitted_On": "train",
                "HMM_Fitted_On": "train",
                "Threshold_Selected_On": "validation",
                "Label_Rule_Note": label_note,
            }
        )

weekly_hmm_robustness_features_df = pd.concat(feature_frames, ignore_index=True)
weekly_hmm_robustness_state_summary_df = pd.DataFrame(state_summary_rows)
weekly_hmm_robustness_transition_df = pd.DataFrame(transition_rows)
weekly_hmm_robustness_expected_duration_df = pd.DataFrame(duration_rows)
weekly_hmm_robustness_model_diagnostics_df = pd.DataFrame(diagnostic_rows)
weekly_hmm_robustness_scaler_df = pd.DataFrame(scaler_rows)

display(weekly_hmm_robustness_model_diagnostics_df)
display(weekly_hmm_robustness_state_summary_df.head(12))


## HMM-Only Benchmarks by Variant

In [ ]:
hmm_only_rows = []
prediction_frames = []
threshold_frames = []

for (ticker, variant), df_group in weekly_hmm_robustness_features_df.groupby(["Ticker", "Variant"], sort=True):
    df_group = df_group.sort_values("Date").copy()
    spec = HMM_VARIANTS[variant]
    diag = weekly_hmm_robustness_model_diagnostics_df.loc[
        (weekly_hmm_robustness_model_diagnostics_df["Ticker"] == ticker)
        & (weekly_hmm_robustness_model_diagnostics_df["Variant"] == variant)
    ].iloc[0]

    val_df = df_group.loc[df_group["Split"] == "validation"].copy()
    test_df = df_group.loc[df_group["Split"] == "test"].copy()
    if min(len(val_df), len(test_df)) == 0:
        raise ValueError(f"{ticker} {variant} has an empty validation or test split.")

    y_val = val_df["Target_State_Binary"].to_numpy(dtype=int)
    y_test = test_df["Target_State_Binary"].to_numpy(dtype=int)
    val_prob = val_df["HMM_Positive_Return_Prob_Next"].to_numpy(dtype=float)
    test_prob = test_df["HMM_Positive_Return_Prob_Next"].to_numpy(dtype=float)

    decision_threshold, threshold_df = select_validation_threshold(y_val, val_prob)
    threshold_df.insert(0, "Ticker", ticker)
    threshold_df.insert(1, "Variant", variant)
    threshold_frames.append(threshold_df)

    val_metrics = binary_metrics(y_val, val_prob, decision_threshold)
    test_metrics = binary_metrics(y_test, test_prob, decision_threshold)
    row = {
        "Ticker": ticker,
        "Variant": variant,
        "n_components": int(spec["n_components"]),
        "feature_set": ", ".join(spec["feature_cols"]),
        "selected_seed": int(diag["selected_seed"]),
        "decision_threshold": decision_threshold,
        "degenerate_prediction_warning": (
            test_metrics["share_predicted_bullish"] <= 0.05
            or test_metrics["share_predicted_bullish"] >= 0.95
        ),
        "low_probability_variation_warning": test_metrics["probability_std"] < LOW_PROBABILITY_STD_CUTOFF,
    }
    row.update({f"val_{key}": value for key, value in val_metrics.items()})
    row.update({f"test_{key}": value for key, value in test_metrics.items()})
    hmm_only_rows.append(row)

    for split_name, split_df, y_true, y_prob in [
        ("validation", val_df, y_val, val_prob),
        ("test", test_df, y_test, test_prob),
    ]:
        prediction_frames.append(
            pd.DataFrame(
                {
                    "Ticker": ticker,
                    "Variant": variant,
                    "Split": split_name,
                    "Date": split_df["Date"].to_numpy(),
                    "Target_Date": split_df["Target_Date"].to_numpy(),
                    "Decision_Threshold": decision_threshold,
                    "y_true": y_true,
                    "y_prob": y_prob,
                    "y_pred": (y_prob >= decision_threshold).astype(int),
                    "HMM_Regime_Label": split_df["HMM_Regime_Label"].to_numpy(),
                    "HMM_State": split_df["HMM_State"].to_numpy(),
                }
            )
        )

weekly_hmm_robustness_hmm_only_results_df = pd.DataFrame(hmm_only_rows).sort_values(["Ticker", "Variant"]).reset_index(drop=True)
weekly_hmm_robustness_hmm_only_predictions_df = pd.concat(prediction_frames, ignore_index=True)
weekly_hmm_robustness_threshold_search_df = pd.concat(threshold_frames, ignore_index=True)

display(weekly_hmm_robustness_hmm_only_results_df[
    [
        "Ticker",
        "Variant",
        "decision_threshold",
        "val_balanced_accuracy",
        "val_roc_auc",
        "test_balanced_accuracy",
        "test_roc_auc",
        "test_f1",
        "test_cohen_kappa",
        "test_share_predicted_bullish",
        "test_probability_std",
        "degenerate_prediction_warning",
        "low_probability_variation_warning",
    ]
])


## Variant Comparison Table

In [ ]:
comparison_cols = [
    "Ticker",
    "Variant",
    "n_components",
    "feature_set",
    "selected_seed",
    "train_log_likelihood",
    "AIC",
    "BIC",
]
weekly_hmm_robustness_variant_comparison_df = (
    weekly_hmm_robustness_model_diagnostics_df[comparison_cols]
    .merge(
        weekly_hmm_robustness_hmm_only_results_df[
            [
                "Ticker",
                "Variant",
                "val_roc_auc",
                "val_balanced_accuracy",
                "test_roc_auc",
                "test_balanced_accuracy",
                "test_f1",
                "test_cohen_kappa",
                "test_share_predicted_bullish",
                "test_share_actual_bullish",
                "degenerate_prediction_warning",
                "low_probability_variation_warning",
            ]
        ],
        on=["Ticker", "Variant"],
        how="left",
        validate="one_to_one",
    )
)

baseline = weekly_hmm_robustness_variant_comparison_df.loc[
    weekly_hmm_robustness_variant_comparison_df["Variant"] == "univariate_2state",
    ["Ticker", "test_balanced_accuracy", "test_roc_auc", "test_f1", "test_cohen_kappa"],
].rename(
    columns={
        "test_balanced_accuracy": "baseline_test_balanced_accuracy",
        "test_roc_auc": "baseline_test_roc_auc",
        "test_f1": "baseline_test_f1",
        "test_cohen_kappa": "baseline_test_cohen_kappa",
    }
)

weekly_hmm_robustness_variant_comparison_df = weekly_hmm_robustness_variant_comparison_df.merge(
    baseline,
    on="Ticker",
    how="left",
    validate="many_to_one",
)
weekly_hmm_robustness_variant_comparison_df["delta_test_balanced_accuracy_vs_univariate"] = (
    weekly_hmm_robustness_variant_comparison_df["test_balanced_accuracy"]
    - weekly_hmm_robustness_variant_comparison_df["baseline_test_balanced_accuracy"]
)
weekly_hmm_robustness_variant_comparison_df["delta_test_roc_auc_vs_univariate"] = (
    weekly_hmm_robustness_variant_comparison_df["test_roc_auc"]
    - weekly_hmm_robustness_variant_comparison_df["baseline_test_roc_auc"]
)
weekly_hmm_robustness_variant_comparison_df["delta_test_f1_vs_univariate"] = (
    weekly_hmm_robustness_variant_comparison_df["test_f1"]
    - weekly_hmm_robustness_variant_comparison_df["baseline_test_f1"]
)
weekly_hmm_robustness_variant_comparison_df["delta_test_kappa_vs_univariate"] = (
    weekly_hmm_robustness_variant_comparison_df["test_cohen_kappa"]
    - weekly_hmm_robustness_variant_comparison_df["baseline_test_cohen_kappa"]
)
weekly_hmm_robustness_variant_comparison_df = weekly_hmm_robustness_variant_comparison_df.drop(
    columns=[
        "baseline_test_balanced_accuracy",
        "baseline_test_roc_auc",
        "baseline_test_f1",
        "baseline_test_cohen_kappa",
    ]
)

display(weekly_hmm_robustness_variant_comparison_df)


## Subperiod and rolling ROC diagnostics

These diagnostics use the HMM-only predictions from each variant. They are not used for model selection.


In [ ]:
rolling_rows = []
subperiod_rows = []

test_predictions_df = weekly_hmm_robustness_hmm_only_predictions_df.loc[
    weekly_hmm_robustness_hmm_only_predictions_df["Split"] == "test"
].copy()
test_predictions_df["Target_Date"] = pd.to_datetime(test_predictions_df["Target_Date"])

for (ticker, variant), df_group in test_predictions_df.groupby(["Ticker", "Variant"], sort=True):
    df_group = df_group.sort_values("Target_Date").reset_index(drop=True)
    for window_size in [52, 104]:
        for end_idx in range(window_size - 1, len(df_group)):
            window = df_group.iloc[end_idx - window_size + 1 : end_idx + 1]
            n_positive = int(window["y_true"].sum())
            n_observations = int(len(window))
            n_negative = n_observations - n_positive
            rolling_auc = (
                roc_auc_score(window["y_true"], window["y_prob"])
                if n_positive > 0 and n_negative > 0
                else np.nan
            )
            rolling_rows.append(
                {
                    "Ticker": ticker,
                    "Variant": variant,
                    "Window_Size": window_size,
                    "Window_End_Date": window["Target_Date"].iloc[-1],
                    "Rolling_ROC_AUC": rolling_auc,
                    "n_observations": n_observations,
                    "n_positive": n_positive,
                    "n_negative": n_negative,
                }
            )

    midpoint = len(df_group) // 2
    subperiods = {
        "first_half_test": df_group.iloc[:midpoint],
        "second_half_test": df_group.iloc[midpoint:],
    }
    for subperiod_name, sub_df in subperiods.items():
        y_true = sub_df["y_true"].to_numpy(dtype=int)
        y_prob = sub_df["y_prob"].to_numpy(dtype=float)
        y_pred = sub_df["y_pred"].to_numpy(dtype=int)
        subperiod_rows.append(
            {
                "Ticker": ticker,
                "Variant": variant,
                "Subperiod": subperiod_name,
                "Start_Date": sub_df["Target_Date"].min(),
                "End_Date": sub_df["Target_Date"].max(),
                "ROC_AUC": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
                "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
                "F1": f1_score(y_true, y_pred, zero_division=0),
                "Cohen_Kappa": cohen_kappa_score(y_true, y_pred),
                "actual_bullish_share": float(np.mean(y_true)),
                "predicted_bullish_share": float(np.mean(y_pred)),
            }
        )

weekly_hmm_robustness_rolling_roc_df = pd.DataFrame(rolling_rows)
weekly_hmm_robustness_subperiod_metrics_df = pd.DataFrame(subperiod_rows)

display(weekly_hmm_robustness_rolling_roc_df.head())
display(weekly_hmm_robustness_subperiod_metrics_df)


## Save Robustness Outputs

In [ ]:
weekly_hmm_robustness_features_df.to_parquet(FEATURES_PATH, index=False)
weekly_hmm_robustness_state_summary_df.to_csv(STATE_SUMMARY_PATH, index=False)
weekly_hmm_robustness_transition_df.to_csv(TRANSITION_PATH, index=False)
weekly_hmm_robustness_expected_duration_df.to_csv(EXPECTED_DURATION_PATH, index=False)
weekly_hmm_robustness_model_diagnostics_df.to_csv(MODEL_DIAGNOSTICS_PATH, index=False)
weekly_hmm_robustness_hmm_only_results_df.to_csv(HMM_ONLY_RESULTS_PATH, index=False)
weekly_hmm_robustness_hmm_only_predictions_df.to_csv(HMM_ONLY_PREDICTIONS_PATH, index=False)
weekly_hmm_robustness_threshold_search_df.to_csv(THRESHOLD_SEARCH_PATH, index=False)
weekly_hmm_robustness_variant_comparison_df.to_csv(VARIANT_COMPARISON_PATH, index=False)
weekly_hmm_robustness_rolling_roc_df.to_csv(ROLLING_ROC_PATH, index=False)
weekly_hmm_robustness_subperiod_metrics_df.to_csv(SUBPERIOD_METRICS_PATH, index=False)

print("Saved robustness outputs:")
for path in EXPECTED_OUTPUT_FILES:
    print(path)


## Research Plots

In [ ]:
variant_order = list(HMM_VARIANTS)
variant_colors = {
    "univariate_2state": "tab:blue",
    "rich_2state": "tab:orange",
    "rich_3state": "tab:green",
}

# Plot 1: Variant comparison by ticker
fig, axes = plt.subplots(1, len(tickers), figsize=(5.8 * len(tickers), 4.4), sharey=True)
if len(tickers) == 1:
    axes = [axes]
bar_width = 0.36
x_positions = np.arange(len(variant_order))
for ax, ticker in zip(axes, tickers):
    df_t = weekly_hmm_robustness_variant_comparison_df.loc[
        weekly_hmm_robustness_variant_comparison_df["Ticker"] == ticker
    ].set_index("Variant").reindex(variant_order)
    ax.bar(x_positions - bar_width / 2, df_t["test_balanced_accuracy"], width=bar_width, label="Test balanced accuracy")
    ax.bar(x_positions + bar_width / 2, df_t["test_roc_auc"], width=bar_width, label="Test ROC-AUC")
    ax.axhline(0.5, color="black", linestyle=":", linewidth=1.2)
    ax.set_title(ticker)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(variant_order, rotation=25, ha="right")
    ax.set_ylim(0, 1)
    ax.grid(True, axis="y", linestyle="--", alpha=0.45)
axes[0].set_ylabel("Score")
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
fig.suptitle("Weekly HMM robustness: test metrics by variant", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.86, 0.93])
save_plot("weekly_hmm_robustness_variant_comparison.png")
plt.show()
plt.close(fig)

# Plot 2: Delta vs univariate HMM
delta_df = weekly_hmm_robustness_variant_comparison_df.loc[
    weekly_hmm_robustness_variant_comparison_df["Variant"] != "univariate_2state"
].copy()
fig, axes = plt.subplots(1, len(tickers), figsize=(5.8 * len(tickers), 4.2), sharey=True)
if len(tickers) == 1:
    axes = [axes]
bar_width = 0.36
x_positions = np.arange(2)
for ax, ticker in zip(axes, tickers):
    df_t = delta_df.loc[delta_df["Ticker"] == ticker].set_index("Variant").reindex(["rich_2state", "rich_3state"])
    ax.bar(x_positions - bar_width / 2, df_t["delta_test_balanced_accuracy_vs_univariate"], width=bar_width, label="Balanced accuracy delta")
    ax.bar(x_positions + bar_width / 2, df_t["delta_test_roc_auc_vs_univariate"], width=bar_width, label="ROC-AUC delta")
    ax.axhline(0, color="black", linestyle=":", linewidth=1.2)
    ax.set_title(ticker)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(["rich_2state", "rich_3state"], rotation=20, ha="right")
    ax.grid(True, axis="y", linestyle="--", alpha=0.45)
axes[0].set_ylabel("Delta versus univariate 2-state HMM")
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
fig.suptitle("Robustness deltas relative to univariate HMM", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.86, 0.93])
save_plot("weekly_hmm_robustness_delta_vs_univariate.png")
plt.show()
plt.close(fig)

# Plot 3: State mean-volatility map
fig, ax = plt.subplots(figsize=(9, 6))
for variant in variant_order:
    df_v = weekly_hmm_robustness_state_summary_df.loc[weekly_hmm_robustness_state_summary_df["Variant"] == variant]
    ax.scatter(
        df_v["Train_Mean_Weekly_Log_Return"],
        df_v["Train_Std_Weekly_Log_Return"],
        s=80,
        alpha=0.75,
        label=variant,
        color=variant_colors[variant],
    )
    for row in df_v.itertuples(index=False):
        ax.annotate(f"{row.Ticker}-{row.HMM_Regime_Label}", (row.Train_Mean_Weekly_Log_Return, row.Train_Std_Weekly_Log_Return), fontsize=8, alpha=0.8)
ax.axvline(0, color="black", linestyle=":", linewidth=1)
ax.set_xlabel("Training mean weekly return")
ax.set_ylabel("Training weekly return volatility")
ax.set_title("HMM state mean-volatility map", fontsize=14, fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.45)
ax.legend(frameon=False)
fig.tight_layout()
save_plot("weekly_hmm_robustness_state_mean_vol_map.png")
plt.show()
plt.close(fig)

# Plot 4: Transition persistence by variant
duration_plot_df = weekly_hmm_robustness_expected_duration_df.copy()
duration_plot_df["Ticker_Variant_Regime"] = (
    duration_plot_df["Ticker"] + "\n" + duration_plot_df["Variant"] + "\n" + duration_plot_df["HMM_Regime_Label"]
)
fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(duration_plot_df["Ticker_Variant_Regime"], duration_plot_df["expected_duration"], alpha=0.9)
ax.set_ylabel("Expected duration in weeks")
ax.set_title("Expected regime durations by ticker and HMM variant", fontsize=14, fontweight="bold")
ax.tick_params(axis="x", rotation=75)
ax.grid(True, axis="y", linestyle="--", alpha=0.45)
fig.tight_layout()
save_plot("weekly_hmm_robustness_expected_durations.png")
plt.show()
plt.close(fig)

# Plot 5: HMM positive-return probability over time
fig, axes = plt.subplots(len(tickers), 1, figsize=(14, 3.5 * len(tickers)), sharex=True, sharey=True)
if len(tickers) == 1:
    axes = [axes]
for ax, ticker in zip(axes, tickers):
    for variant in variant_order:
        df_t = weekly_hmm_robustness_features_df.loc[
            (weekly_hmm_robustness_features_df["Ticker"] == ticker)
            & (weekly_hmm_robustness_features_df["Variant"] == variant)
            & (weekly_hmm_robustness_features_df["Split"] == "test")
        ].sort_values("Target_Date")
        ax.plot(df_t["Target_Date"], df_t["HMM_Positive_Return_Prob_Next"], linewidth=1.6, label=variant, color=variant_colors[variant])
    ax.set_title(ticker)
    ax.set_ylabel("Probability")
    ax.set_ylim(0, 1)
    ax.grid(True, linestyle="--", alpha=0.45)
    ax.legend(loc="upper right")
axes[-1].set_xlabel("Target week")
fig.suptitle("HMM positive-return probabilities over the test period", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
save_plot("weekly_hmm_robustness_positive_return_prob_over_time.png")
plt.show()
plt.close(fig)

# Plot 6: HMM-only ROC curves by variant
fig, axes = plt.subplots(1, len(tickers), figsize=(5.8 * len(tickers), 4.8), sharey=True)
if len(tickers) == 1:
    axes = [axes]
for ax, ticker in zip(axes, tickers):
    for variant in variant_order:
        df_t = test_predictions_df.loc[(test_predictions_df["Ticker"] == ticker) & (test_predictions_df["Variant"] == variant)]
        if df_t["y_true"].nunique() < 2:
            continue
        fpr, tpr, _ = roc_curve(df_t["y_true"], df_t["y_prob"])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=2, label=f"{variant} AUC={roc_auc:.3f}", color=variant_colors[variant])
    ax.plot([0, 1], [0, 1], color="black", linestyle=":", linewidth=1.2)
    ax.set_title(ticker)
    ax.set_xlabel("False positive rate")
    ax.grid(True, linestyle="--", alpha=0.45)
axes[0].set_ylabel("True positive rate")
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
fig.suptitle("HMM-only test ROC curves by robustness variant", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.82, 0.93])
save_plot("weekly_hmm_robustness_roc_curves_by_variant.png")
plt.show()
plt.close(fig)


In [ ]:
# Plot 7 and 8: Rolling ROC-AUC
for window_size, filename in [
    (52, "weekly_hmm_robustness_rolling_roc_52.png"),
    (104, "weekly_hmm_robustness_rolling_roc_104.png"),
]:
    fig, axes = plt.subplots(len(tickers), 1, figsize=(14, 3.4 * len(tickers)), sharex=True, sharey=True)
    if len(tickers) == 1:
        axes = [axes]
    for ax, ticker in zip(axes, tickers):
        for variant in variant_order:
            df_t = weekly_hmm_robustness_rolling_roc_df.loc[
                (weekly_hmm_robustness_rolling_roc_df["Ticker"] == ticker)
                & (weekly_hmm_robustness_rolling_roc_df["Variant"] == variant)
                & (weekly_hmm_robustness_rolling_roc_df["Window_Size"] == window_size)
            ].sort_values("Window_End_Date")
            ax.plot(df_t["Window_End_Date"], df_t["Rolling_ROC_AUC"], linewidth=1.6, label=variant, color=variant_colors[variant])
        ax.axhline(0.5, color="black", linestyle=":", linewidth=1.1)
        ax.set_ylim(0, 1)
        ax.set_title(ticker)
        ax.set_ylabel("Rolling ROC-AUC")
        ax.grid(True, linestyle="--", alpha=0.45)
        ax.legend(loc="upper right")
    axes[-1].set_xlabel("Window end date")
    fig.suptitle(f"Rolling {window_size}-week HMM-only ROC-AUC", fontsize=14, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save_plot(filename)
    plt.show()
    plt.close(fig)

# Plot 9: Subperiod ROC-AUC
fig, axes = plt.subplots(1, len(tickers), figsize=(6.2 * len(tickers), 4.5), sharey=True)
if len(tickers) == 1:
    axes = [axes]
bar_width = 0.12
subperiods = ["first_half_test", "second_half_test"]
base_positions = np.arange(len(variant_order))
offsets = [-bar_width / 2, bar_width / 2]
for ax, ticker in zip(axes, tickers):
    df_t = weekly_hmm_robustness_subperiod_metrics_df.loc[weekly_hmm_robustness_subperiod_metrics_df["Ticker"] == ticker]
    for offset, subperiod in zip(offsets, subperiods):
        values = (
            df_t.loc[df_t["Subperiod"] == subperiod]
            .set_index("Variant")
            .reindex(variant_order)["ROC_AUC"]
        )
        ax.bar(base_positions + offset, values, width=bar_width, label=subperiod, alpha=0.9)
    ax.axhline(0.5, color="black", linestyle=":", linewidth=1.1)
    ax.set_title(ticker)
    ax.set_xticks(base_positions)
    ax.set_xticklabels(variant_order, rotation=25, ha="right")
    ax.set_ylim(0, 1)
    ax.grid(True, axis="y", linestyle="--", alpha=0.45)
axes[0].set_ylabel("Subperiod ROC-AUC")
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
fig.suptitle("First-half versus second-half test ROC-AUC", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.84, 0.93])
save_plot("weekly_hmm_robustness_subperiod_roc.png")
plt.show()
plt.close(fig)

print("Saved robustness plots:")
for path in saved_plot_paths:
    print(path)


## Final Sanity Checks

In [ ]:
if len(HMM_VARIANTS) != 3:
    raise AssertionError(f"Expected exactly three HMM variants, got {len(HMM_VARIANTS)}.")

expected_variant_names = set(HMM_VARIANTS)
actual_variant_names = set(weekly_hmm_robustness_model_diagnostics_df["Variant"])
if actual_variant_names != expected_variant_names:
    raise AssertionError(f"Unexpected fitted variants: {actual_variant_names} != {expected_variant_names}")

expected_fit_count = len(tickers) * len(HMM_VARIANTS)
if len(weekly_hmm_robustness_model_diagnostics_df) != expected_fit_count:
    raise AssertionError(
        f"Expected {expected_fit_count} ticker-variant HMM fits, got {len(weekly_hmm_robustness_model_diagnostics_df)}."
    )

if not weekly_hmm_robustness_scaler_df["Fitted_On"].eq("train").all():
    raise AssertionError("At least one scaler was not marked as fitted on training rows only.")
if not weekly_hmm_robustness_model_diagnostics_df["HMM_Fitted_On"].eq("train").all():
    raise AssertionError("At least one HMM was not marked as fitted on training rows only.")
if not weekly_hmm_robustness_model_diagnostics_df["Threshold_Selected_On"].eq("validation").all():
    raise AssertionError("At least one threshold was not marked as selected on validation rows only.")

if not weekly_hmm_robustness_features_df["HMM_Positive_Return_Prob_Next"].between(0, 1).all():
    raise AssertionError("HMM_Positive_Return_Prob_Next must be between 0 and 1.")

two_state_df = weekly_hmm_robustness_features_df.loc[weekly_hmm_robustness_features_df["n_components"] == 2].copy()
two_state_sum = two_state_df["HMM_Bullish_Prob_Next"] + two_state_df["HMM_Bearish_Prob_Next"]
if not np.allclose(two_state_sum, 1.0, atol=1e-6):
    raise AssertionError("For 2-state HMMs, bullish + bearish next probabilities must sum to 1.")

three_state_df = weekly_hmm_robustness_features_df.loc[weekly_hmm_robustness_features_df["n_components"] == 3].copy()
three_state_sum = (
    three_state_df["HMM_State_0_Prob_Next"]
    + three_state_df["HMM_State_1_Prob_Next"]
    + three_state_df["HMM_State_2_Prob_Next"]
)
if not np.allclose(three_state_sum, 1.0, atol=1e-6):
    raise AssertionError("For 3-state HMMs, generic next probabilities must sum to 1.")

missing_output_files = [path for path in EXPECTED_OUTPUT_FILES if not path.exists()]
if missing_output_files:
    raise FileNotFoundError(f"Missing required robustness output files: {missing_output_files}")

missing_plot_files = [path for path in EXPECTED_PLOT_FILES if not path.exists()]
if missing_plot_files:
    raise FileNotFoundError(f"Missing required robustness plot files: {missing_plot_files}")

sanity_df = pd.DataFrame(
    [
        {"Check": "Exactly three HMM variants were tested", "Passed": True},
        {"Check": "Each variant was fitted separately for each ticker", "Passed": True},
        {"Check": "Scalers were fitted on training rows only", "Passed": True},
        {"Check": "HMMs were fitted on training rows only", "Passed": True},
        {"Check": "No test data was used for fitting or threshold selection", "Passed": True},
        {"Check": "HMM_Positive_Return_Prob_Next is between 0 and 1", "Passed": True},
        {"Check": "2-state next probabilities sum to 1", "Passed": True},
        {"Check": "3-state next probabilities sum to 1", "Passed": True},
        {"Check": "All required CSV/parquet files exist", "Passed": True},
        {"Check": "All required plot files exist", "Passed": True},
    ]
)
display(sanity_df)

print("Robustness HMM fits:", len(weekly_hmm_robustness_model_diagnostics_df))
print("Robustness feature rows:", len(weekly_hmm_robustness_features_df))
print("Expected plot files:", len(EXPECTED_PLOT_FILES))
print("All weekly HMM robustness sanity checks passed.")


## Final Interpretation

This notebook tests whether the weak incremental signal from the original HMM is due to the univariate weekly-return specification. The original two-state HMM is treated as the interpretable baseline. The rich two-state HMM adds volatility, momentum, and drawdown to test whether a broader description of market conditions improves the regime signal. The rich three-state HMM tests whether the market is better represented by more than two regimes, such as calm, transition, and stress regimes.

The results should be interpreted cautiously. A richer HMM is useful only if it improves validation and test metrics without becoming degenerate or unstable. Rolling ROC-AUC and subperiod results are diagnostic only; they help show whether the model works better in some market periods, but they should not be used to select the final model after looking at the test set.

If the richer HMM improves the HMM-only benchmark and gives more variable, interpretable regime probabilities, it can be considered as a future extension for the HMM-informed LSTM. If it does not improve materially, the conclusion is that HMM regimes are interpretable but provide limited incremental directional forecasting power in this dataset.
